In [1]:
import pypeline
import numpy as np
from glob import glob
import os
import sys
import mne_bids
from contextlib import contextmanager
import mne
from datetime import datetime
%load_ext autoreload

%autoreload 2

In [2]:
EXPERIMENT_NAME = "banyan2"  # name of the experiment
parent_dir = "../raw_data"  # directory where your raw data (folder containing brainvision, eyetracking asc, and behavior is stored)
data_dir = "../data"  # where to output data
if not os.path.exists(data_dir):
    os.makedirs(data_dir)
file_prefix = ""  # prefix to your vhdr files. Assuming it is in the format [prefix][number]

overwrite_subs = False  # if you want to overwrite the data for a subject, set to True
subject_dirs = [
    "01",
    "02",
    "03",
    "04",
    "05",
    "06",
    "07",
    "08",
    "09",
    "10",
    "11",
    "12",
    "13",
    "14",
    "15",
    "16",
    "17",
    "18",
    "19",
    "20",
    "21",
    "22",
    "23",
    "25",
    "26",
    "27",
    "28",
    "30",
    "31",
    "32",
    "33",
    "34",
    "35",
    "36",
    ]  # if you want to analyze a specific subset of subjects
if len(subject_dirs) == 0:
    subject_dirs = sorted(glob("*", root_dir=parent_dir))
if not overwrite_subs:
    subject_dirs = [sub for sub in subject_dirs if sub not in [f[4:] for f in glob("sub-*", root_dir=data_dir)]]

print("Subject to analyze: ", subject_dirs)


TRIAL_START_TIME = -0.45  # epoch start before your designated timelock code
TRIAL_END_TIME = 1.4
BASELINE_TIME = (-0.35, 0)  # time for baseline correction

SRATE = 500  # hz, will resample if different from the raw data srate
FILTER_FREQS = (None, 50)  # None to not do one of the filtering steps

LINEAR_R2 = 0.3

event_dict = {  # this should be a dict of names of ALL the event codes that appear
    "trl_start": 1,
    "attn/ss0": 10,
    "attn/ss1": 11,
    "attn/ss2": 12,
    "attn/ss3": 13,
    "attn/ss4": 14,
    "attn/ss6": 16,
    "attn/ss8": 18,
    "wm/ss0": 20,
    "wm/ss1": 21,
    "wm/ss2": 22,
    "wm/ss3": 23,
    "wm/ss4": 24,
    "wm/ss6": 26,
    "wm/ss8": 28,
    "delay_start": 2,
    "attention_probe": 3,  # if probe attention, the code seq is 2 3 2 4
    "delay_end": 4,
    "response": 5,
}
# event_dict is a list of name: number pairings for all the TRIAL event codes
# event_code_dict: a dict of code: sequence pairings for each trial
# so, if you have a trial with fixation (1) -> SS2 stimulus (12) -> delay (3) -> test (4), that you want to map to code  12:
# {12 : [1,12,3,4]}... and so on
event_code_dict = {}  # define event codes based on sequence

for key, ev in event_dict.copy().items():
    if ev >= 10:
        event_dict.update({key + "/PROBE": ev + 20})  # add in keys for targets
        event_code_dict.update({ev: [1, ev, 2, 4]})
        event_code_dict.update({ev + 20: [1, ev, 2, 3, 2, 4]})


POSITION_TO_TIMELOCK = 1  # which position (IN THE LIST ABOVE) to timelock to. Should be the one which indicates the condition.

EYEKEYWORD = "SYNC"

# edge case when we forgot to start the recording or recordings crashed for one but not the another, manually drop certain trials
# Must be in the form of {'subject number':[list of ints]}
EEG_TRIALS_DROP = {"05": [737, 739, 740, 741, 745, 746]}
EYE_TRIALS_DROP = {"05": [738, 739, 743], "21": [649, 650, 651, 652]}
# channels to ignore when rejecting trials. Will also be excluded from analysis.
SUB_IGNORE_CHANS = {}

NO_EYES = []  # subjects with no eye data
DROP_CHANNELS = (
    []
)  # channels to delete from the dataset entirely. Recommendation is to leave this blank and instead set REJ_TRIALS_IGNORE later on

Subject to analyze:  []


In [3]:
pre = pypeline.Preprocess(
    data_dir=data_dir,
    root_dir=parent_dir,
    experiment_name=EXPERIMENT_NAME,
    srate=SRATE,
    trial_start=TRIAL_START_TIME,
    trial_end=TRIAL_END_TIME,
    event_dict=event_dict,
    event_code_dict=event_code_dict,
    timelock_ix=POSITION_TO_TIMELOCK,
    baseline_time=BASELINE_TIME,
    rejection_time=None,
    reject_between_codes=[1, 4],
    no_et_spaces=False,
    drop_channels=DROP_CHANNELS,
    filter_freqs=FILTER_FREQS,
)

In [4]:
@contextmanager
def redirect_stdout(new_stdout):  # writes the output to a log file
    save_stdout = sys.stdout
    save_stderr = sys.stderr
    sys.stdout = new_stdout
    sys.stderr = sys.stdout
    try:
        yield None
    finally:
        sys.stdout = save_stdout
        sys.stderr = save_stderr


# with open('preprocessing_log.txt','a+') as f:
#     with redirect_stdout(f):

print("\n\n\n##########################\n" + "STARTING PREPROCESSING RUN\n" + "##########################\n\n\n")
print(f'Run started at {datetime.now().strftime("%H:%M:%S")}')
for subject_number in subject_dirs:

    print(
        "\n\n#############################\n"
        + f"## STARTING NEW SUBJECT {subject_number} ##\n"
        + "#############################\n"
    )

    #####################
    #### IMPORT DATA ####
    #####################

    # import into the RAW bids dataset
    eeg, eeg_events = pre.import_eeg(subject_number, overwrite=True)
    reref_index = mne.pick_channels(eeg.ch_names, ["TP9"])
    eeg.load_data().apply_function(
        pre.rereference_to_average, picks=["eeg"], reref_values=np.squeeze(eeg.get_data()[reref_index])
    )
    eeg.filter(*pre.filter_freqs, n_jobs=-1)

    pre.import_behavior(subject_number)

    ########################################
    #### PREPROCESS EEG AND MAKE EPOCHS ####
    ########################################

    if subject_number in NO_EYES:
        epochs = pre.make_eeg_epochs(eeg, eeg_events, eeg_trials_drop=EEG_TRIALS_DROP.get(subject_number, None))
    else:
        eye, eye_events = pre.import_eyetracker(subject_number, keyword=EYEKEYWORD)

        epochs = pre.make_and_sync_epochs(
            eeg,
            eeg_events,
            eye,
            eye_events,
            eeg_trials_drop=EEG_TRIALS_DROP.get(subject_number, None),
            eye_trials_drop=EYE_TRIALS_DROP.get(subject_number, None),
        )

    ###############################
    #### DO ARTIFACT REJECTION ####
    ###############################

    p2p = pre.artreject_slidingP2P(
        epochs, rejection_criteria={"eeg": 100e-6, "eog": 200}, win=200, win_step=100
    )  # peak to peak in the window
    saccades = pre.artreject_step(
        epochs, rejection_criteria={"eyegaze": pre.deg2pix(0.5), "eog": 50}, win=80, win_step=10
    )  # saccades in EOG or eye tracking
    steps = pre.artreject_step(
        epochs, rejection_criteria={"eeg": 60e-6}, win=250, win_step=20
    )  # steps (saccade like) in EEG

    absolute_value = pre.artreject_value(
        epochs, rejection_criteria={"eyegaze": pre.deg2pix(1), "eeg": 100e-6, "eog": 300}
    )  # absolute value rejection
    linear_fit = pre.artreject_linear(epochs)  # linear fit (drift) rejection
    flatline = pre.artreject_flatline(
        epochs,
        rejection_criteria={"eeg": 0, "eog": 0, "eyegaze": 0},
        flatline_duration=200,
    )  # check for flatlines

    # combine rejection reasons
    rej_electrodes = p2p | saccades | steps | absolute_value | linear_fit | flatline
    rej_reasons = np.char.array(
        np.full(rej_electrodes.shape, "", dtype="<U30")
    )  # NOTE: dtype is important, must be >= the max possible str length
    rej_reasons[p2p] = "P2P "
    rej_reasons[saccades] = rej_reasons[saccades] + "SAC "
    rej_reasons[steps] = rej_reasons[steps] + "STEP "
    rej_reasons[absolute_value] = rej_reasons[absolute_value] + "ABS "
    rej_reasons[linear_fit] = rej_reasons[linear_fit] + "LIN "
    rej_reasons[flatline] = rej_reasons[flatline] + "FLAT "

    rej_counts = lambda x: f"{x.any(1).sum()} ({round(x.any(1).sum() / x.shape[0] * 100,1)}%)"
    print(
        (
            f"Rejected {rej_electrodes.any(1).sum()} trials ({round(rej_electrodes.any(1).sum() / rej_electrodes.shape[0] * 100,1)}%) for the following reasons:\n"
            f"Peak to peak amplitude: {rej_counts(p2p)}\n"
            f"Saccades: {rej_counts(saccades)}\n"
            f"Steps: {rej_counts(steps)}\n"
            f"Absolute value: {rej_counts(absolute_value)}\n"
            f"Linear fit: {rej_counts(linear_fit)}\n"
            f"Flatline: {rej_counts(flatline)}\n"
        )
    )

    print(
        "Worst electrodes by count:\n"
        + "\n".join(
            [f"{epochs.ch_names[i]}: {rej_electrodes[:,i].sum()}" for i in np.argsort(rej_electrodes.sum(0))[::-1][0:5]]
        )
    )

    #################################
    #### SAVE DATA AS DERIVATIVE ####
    #################################

    pre.save_all_data(subject_number, epochs, rej_reasons)
print(mne_bids.make_report(data_dir))




##########################
STARTING PREPROCESSING RUN
##########################



Run started at 15:44:14
Summarizing participants.tsv ../data/participants.tsv...
Summarizing scans.tsv files [PosixPath('../data/sub-13/sub-13_scans.tsv'), PosixPath('../data/sub-14/sub-14_scans.tsv'), PosixPath('../data/sub-22/sub-22_scans.tsv'), PosixPath('../data/sub-25/sub-25_scans.tsv'), PosixPath('../data/sub-23/sub-23_scans.tsv'), PosixPath('../data/sub-15/sub-15_scans.tsv'), PosixPath('../data/sub-12/sub-12_scans.tsv'), PosixPath('../data/sub-30/sub-30_scans.tsv'), PosixPath('../data/sub-08/sub-08_scans.tsv'), PosixPath('../data/sub-01/sub-01_scans.tsv'), PosixPath('../data/sub-06/sub-06_scans.tsv'), PosixPath('../data/sub-07/sub-07_scans.tsv'), PosixPath('../data/sub-09/sub-09_scans.tsv'), PosixPath('../data/sub-36/sub-36_scans.tsv'), PosixPath('../data/sub-31/sub-31_scans.tsv'), PosixPath('../data/sub-17/sub-17_scans.tsv'), PosixPath('../data/sub-28/sub-28_scans.tsv'), PosixPath('../data/su

In [5]:
# eeg_epochs, eye_epochs, eeg_events_prepped, eye_events_prepped = pre._make_and_prep_sync_of_epochs(
#     eeg,
#     eeg_events,
#     eye,
#     eye_events,
#     eeg_trials_drop=EEG_TRIALS_DROP.get(subject_number, None),
#     eye_trials_drop=EYE_TRIALS_DROP.get(subject_number, None),
# )


# eeg_trial_codes = eeg_events_prepped[:, 2]
# eeg_trial_codes = eeg_trial_codes[eeg_trial_codes >= 10]
# eye_trial_codes = eye_events_prepped[:, 2]

# bad_eye_trials = np.array([649, 650, 651, 652])
# keep_eye_trials = np.array([i for i in range(len(eye_trial_codes)) if i not in bad_eye_trials])

# import seaborn as sns
# import matplotlib.pyplot as plt

# trial_range = np.arange(len(eeg_trial_codes))  # np.arange(640, 700)

# sns.lineplot(x=trial_range, y=eeg_trial_codes[trial_range], label="EEG")
# sns.lineplot(x=trial_range, y=eye_trial_codes[keep_eye_trials][trial_range], label="Eye Tracking")
# plt.show()

# print(eeg_trial_codes[trial_range], "\n", eye_trial_codes[keep_eye_trials][trial_range])

In [18]:
%matplotlib qt
sub = input('Enter subject number: ') # you can also just set this to a string

REJ_CHANNELS_IGNORE=['StimTrak', 'HEOG','VEOG','TP9', 'Fp1', 'Fp2'] # exclude 'Fp1' and 'Fp2' here if you want to keep trials where they are noisy, and drop them before analysis later


viz = pypeline.Visualizer(sub,
                            parent_dir = data_dir,
                            load_flags = False,  # we're going to rebuild our rejection flags every time
                            experiment_name=EXPERIMENT_NAME,
                            srate=SRATE,
                            rejection_time=[None, None],
                            downscale={'eyegaze':1e-6,'misc':1,'eeg':1,'eog':1e-6}, # convert to equivalent units (probably uV)
                            channels_drop=['pupil_left','pupil_right'],
                            channels_ignore=REJ_CHANNELS_IGNORE + SUB_IGNORE_CHANS.get(sub, []))


rejection_sums = viz.rej_chans.sum(axis=0)
sort_ix = np.argsort(rejection_sums)[::-1]

for ichan,chan in enumerate(viz.chan_labels[sort_ix]):
    if rejection_sums[sort_ix][ichan] > 0:
        print(chan,rejection_sums[sort_ix][ichan])
viz.preprocess_data_for_plot()
viz.open_figure()

Reading /Users/henryjones/Documents/research/banyan/Experiment 2/preprocessing_pypeline/../data/derivatives/sub-36/eeg/sub-36_task-banyan2_desc-preprocessed_eeg.fif ...
    Found the data of interest:
        t =    -450.00 ...    1400.00 ms
        0 CTF compensation matrices available


/Users/henryjones/Documents/research/banyan/Experiment 2/preprocessing_pypeline/pypeline/visualizer.py:57: RuntimeWarning: This filename (../data/derivatives/sub-36/eeg/sub-36_task-banyan2_desc-preprocessed_eeg.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  self.epochs_obj = mne.read_epochs(self.data_path.fpath)


Adding metadata with 18 columns
1680 matching events found
No baseline correction applied
0 projection items activated
Pz 182
PO3 163
PO4 138
Cz 134
FC2 126
P3 124
CP1 121
CP2 120
P4 115
FC1 115
Fz 109
F4 109
C4 99
F3 92
C3 90
CP6 86
PO8 76
FC6 66
PO7 63
CP5 61
O1 53
O2 49
Oz 49
F8 48
xpos_left 41
xpos_right 41
FC5 35
P8 33
P7 31
F7 17
ypos_right 15
ypos_left 15


key not recognized: alt. Press h for help.
key not recognized: alt+control. Press h for help.
138/1680 trials rejected. Saving annotations as "../data/derivatives/sub-36/eeg/sub-36_task-banyan2_desc-preprocessed_rejection_flags.npy"
138/1680 trials rejected. Saving annotations as "../data/derivatives/sub-36/eeg/sub-36_task-banyan2_desc-preprocessed_rejection_flags.npy"


# Move behavior files

In [ ]:
break

In [14]:
from glob import glob
import shutil
import os

In [18]:
sub_dirs = sorted(glob("../data/derivatives/sub-*"))
sub_dirs
for sub_dir in sub_dirs:
    sub_num = sub_dir.split('-')[-1]
    # get all files that end with digits, and not strings
    # exclude those with "-" in the filename, as those are practice blocks
    beh_files = glob(f"../raw_data/{sub_num}/{sub_num}_data_block*[0-9].csv")
    beh_files = [f for f in beh_files if "-" not in f]
    for bf in beh_files:
        # move to data/derivatives/sub-XX/beh/
        new_dir = f"../data/derivatives/sub-{sub_num}/beh/"
        if not os.path.exists(new_dir):
            os.makedirs(new_dir)
        new_file = os.path.join(new_dir, os.path.basename(bf))
        shutil.copy(bf, new_file)

    # copy the effort file
    effort_file = f"../raw_data/{sub_num}/{sub_num}_effort.json"
    if not os.path.exists(effort_file):
        print(f"Effort file not found for subject {sub_num}")
        continue
    new_effort_file = f"../data/derivatives/sub-{sub_num}/beh/{sub_num}_effort.json"
    shutil.copy(effort_file, new_effort_file)